# DEEPX Tutorial 01 - How to Install the DEEPX SDK
This first tutorial explains how to install the DEEPX SDK and verify that the setup is successful. You will learn how to prepare the environment, install the SDK, and confirm that your DEEP NPU devices (DX-M1, DX-M1M, and DX-M1 Quattro) are properly recognized in your system.

After completing this tutorial, you will be able to install DX-All Suite successfully and run a basic flow on the DEEPX NPU devices.

## Recommended system requirements for this tutorial:

`Note:` these requirements are for the tutorial only and are **not mandatory for using DEEPX products.**
- OS: Linux (Ubuntu 20.04/22.04/24.04/26.04, Debian 12/13)
- RAM: 8G (16G for DX-Compiler)
- Storage: Higher than 40G
- DEEPX NPU: DX-M1, DX-M1M, and DX-H1 Quattro
- CPU: DX-Compiler + DX-Runtime on x86_64, DX-Runtime only on aarch64

## DXNN® - DEEPX NPU SDK Introduction (DX-AS: DX-All Suite)

DX-AS (DX-All Suite) is an integrated environment of frameworks and tools that enables inference and compilation of AI models using DEEPX devices. Users can build the integrated environment by installing individual tools, but DX-AS maintains optimal compatibility by aligning the versions of the individual tools.

![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/resources/dxnn_sdk_illustration.png)

The DEEPX SDK is primarily divided into two key parts.

The first is the AI Model Compile Environment part, which transforms your AI model into an optimized format for efficient execution on the DEEPX NPU.

The second is the AI Model Runtime Environment part, which executes the compiled AI model on the actual DEEPX NPU hardware to generate results.

By using DX-All Suite, you can set up both components seamlessly, without managing them individually.

![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/img/dx-as.png)


![](https://github.com/DEEPX-AI/dx-all-suite/raw/main/docs/source/img/DXNN-SDK-Simple-Architecture.png)



For easier understanding, there are two YouTube videos on DX-SDK:
- [Youtube - DEEPX SDK Introduction](https://www.youtube.com/watch?v=Js6Soex0WI4) | Download: [EN](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_ENG.mp4) · [中文](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_CH.mp4) · [한국어](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/1.DXNN_Introduce_KO.mp4)

- [Youtube - Quick Start Guide for DX-All Suite](https://www.youtube.com/watch?v=UXmVeYO5r3c) | Download: [EN](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/2.SDK_Quick_Start_Guide_CH.mp4) · [中文](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/2.SDK_Quick_Start_Guide_CH.mp4) · [한국어](https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/2.SDK_Quick_Start_Guide_KO.mp4)

<img src="assets/youtube-dx-sdk.png" style="max-width: 1000px;">

## 1. Download DX-All Suite

You can download DX-All Suite package from the following git repository:
- https://github.com/DEEPX-AI/dx-all-suite

### Configure the installation

Set the installation directory and Git branch in the next two code cells. These are the only values you need to edit. The notebook only inspects the selected path and prints commands; run clone and installation commands in a separate terminal.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess

root_candidates = []
if os.environ.get("ROOT_PATH"):
    root_candidates.append(Path(os.environ["ROOT_PATH"]))
root_candidates.extend([Path.cwd(), *Path.cwd().parents])

TUTORIAL_ROOT = next(
    (path.resolve() for path in root_candidates if (path / "config.json").is_file()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Could not find dx-tutorials/config.json")

CONFIG_PATH = TUTORIAL_ROOT / "config.json"
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

#### 1) Installation directory

Change the path below if you do not want to use the default location.

In [ ]:
# Edit this value to choose the DX-All Suite installation directory.
DX_ALL_SUITE_DIR = Path("~/dx-all-suite").expanduser().resolve()

print(f"DX-All Suite directory: {DX_ALL_SUITE_DIR}")

#### 2) Git branch

Change the branch name below only when you need another SDK version.

In [ ]:
# Edit this value to choose the DX-All Suite Git branch.
DX_ALL_SUITE_BRANCH = "main-v2.3.3"

print(f"DX-All Suite Git branch: {DX_ALL_SUITE_BRANCH}")

In [ ]:
from IPython.display import Markdown, display

DX_COMPILER_DIR = DX_ALL_SUITE_DIR / "dx-compiler"
DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"
DXCOM_PATH = DX_COMPILER_VENV / "bin" / "dxcom"
DX_COM_DIR = DX_COMPILER_DIR / "dx_com"
DX_RUNTIME_DIR = DX_ALL_SUITE_DIR / "dx-runtime"

print(f"Tutorial root:       {TUTORIAL_ROOT}")
print(f"Configuration file:  {CONFIG_PATH}")
print(f"DX-All Suite path:   {DX_ALL_SUITE_DIR}")
print(f"Git branch:          {DX_ALL_SUITE_BRANCH}")
print(f"Jupyter environment: {os.environ.get('VIRTUAL_ENV', 'not set')}")
print(f"Jupyter Python:      {os.sys.executable}")
print(f"DX-Compiler Python:  {DX_COMPILER_VENV / 'bin' / 'python'}")

def command_output(command):
    result = subprocess.run(command, text=True, capture_output=True, check=False)
    return result.returncode, result.stdout.strip(), result.stderr.strip()

repo_exists = (DX_ALL_SUITE_DIR / ".git").is_dir()
path_conflict = DX_ALL_SUITE_DIR.exists() and not repo_exists
remote_url = ""
current_branch = ""
current_commit = ""
expected_commit = ""
branch_matches = False
submodules_ready = False

if repo_exists:
    _, remote_url, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "remote", "get-url", "origin"]
    )
    _, current_branch, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "branch", "--show-current"]
    )
    _, current_commit, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "rev-parse", "HEAD"]
    )
    _, expected_commit, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "rev-parse", f"{DX_ALL_SUITE_BRANCH}^{{commit}}"]
    )
    branch_matches = bool(current_commit) and current_commit == expected_commit
    status, submodule_output, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "submodule", "status", "--recursive"]
    )
    submodules_ready = status == 0 and all(
        not line.startswith("-") for line in submodule_output.splitlines()
    )
    if "DEEPX-AI/dx-all-suite" not in remote_url:
        path_conflict = True

compiler_installed = DXCOM_PATH.is_file()
runtime_cli = shutil.which("dxrt-cli")
runtime_installed = runtime_cli is not None

print(f"[{'OK' if repo_exists else 'MISSING':7}] Git repository")
if repo_exists:
    print(f"          remote: {remote_url or 'unknown'}")
    print(f"          branch: {current_branch or 'detached or unknown'}")
branch_status = "OK" if branch_matches else ("MISMATCH" if repo_exists else "MISSING")
print(f"[{branch_status:8}] Git branch {DX_ALL_SUITE_BRANCH}")
print(f"[{'OK' if submodules_ready else 'MISSING':7}] Git submodules")
print(f"[{'OK' if compiler_installed else 'MISSING':7}] DX-Compiler environment")
print(f"[{'OK' if runtime_installed else 'MISSING':7}] DX-Runtime CLI")

if path_conflict:
    print("\n[ERROR] The selected path exists but is not the expected DX-All Suite repository.")
    print("Choose another DX_ALL_SUITE_DIR or inspect the existing directory manually.")
else:
    shell_commands = []
    installer_commands = []
    quoted_dir = shlex.quote(str(DX_ALL_SUITE_DIR))
    quoted_branch = shlex.quote(DX_ALL_SUITE_BRANCH)
    if not repo_exists:
        shell_commands.append(
            "git clone --recurse-submodules --branch "
            f"{quoted_branch} https://github.com/DEEPX-AI/dx-all-suite.git {quoted_dir}"
        )
    elif not submodules_ready:
        shell_commands.append(
            f"git -C {quoted_dir} submodule update --init --recursive"
        )
    if repo_exists and not branch_matches:
        print("\n[WARNING] The existing repository is not at the configured Git branch.")
        print("Review local changes and switch to the expected branch manually before verification.")
    if not compiler_installed:
        installer_commands.append("./dx-compiler/install.sh")
    if not runtime_installed:
        installer_commands.append("./dx-runtime/install.sh --all")
    if installer_commands:
        shell_commands.append(f"cd {quoted_dir}")
        shell_commands.extend(installer_commands)

    if shell_commands:
        guide = (
            "### Run in a separate terminal\n\n"
            "Copy and run these Linux commands:\n\n"
            "```bash\n" + "\n".join(shell_commands) + "\n```"
        )
        if not runtime_installed:
            guide += "\n\nA system reboot is required after the NPU driver installation."
        guide += "\n\nAfter installation, return here and run this status cell again."
        display(Markdown(guide))
    else:
        display(Markdown(
            "### Existing installation detected\n\n"
            "Clone and installation steps are not required."
        ))

The DX-All Suite consists of **two primary components**:
 - `DX-Compiler`: Converting from ONNX to DXNN
 - `DX-Runtime`: Running the compiled AI model on the actual DEEPX NPU hardware to generate results

In [ ]:
if not repo_exists:
    print("Clone DX-All Suite in a terminal, then rerun the status cell above.")
elif shutil.which("tree"):
    subprocess.run(["tree", "-L", "1", str(DX_ALL_SUITE_DIR)], check=False)
else:
    print("Optional command 'tree' is missing. Install it with: sudo apt install -y tree")

DX-Runtime provides:
 - `DX-APP`: DEEPX User's Application Templates to show how to use DX NPU in user app perspective
 - `DX-FW`: NPU F/W binary
 - `DX-RT`: Framework designed for optimized execution of inference tasks using DX NPU
 - `DX-NPU Driver`: Linux kernel driver for DX NPU
 - `DX-STREAM`: GStreamer-based Vision AI application development tools for DX NPU

In [ ]:
if DX_RUNTIME_DIR.is_dir() and shutil.which("tree"):
    subprocess.run(["tree", "-L", "1", str(DX_RUNTIME_DIR)], check=False)
else:
    print(f"DX-Runtime source directory not ready: {DX_RUNTIME_DIR}")

## 2. Install DX-Compiler

For more details, see the [DX-All Suite installation guide](https://github.com/DEEPX-AI/dx-all-suite/blob/main-v2.3.3/docs/source/installation.md).

The DX-Compiler environment provides prebuilt binary outputs and does not include source code. Each
module can be downloaded and installed from a remote server using the following command:

- `./dx-compiler/install.sh`

- `Note:` If you want to install dx-compiler or dxtron individually, run the following command:
    - `./dx-compiler/install.sh --target=dx_com`
    - `./dx-compiler/install.sh --target=dx_tron`

### 2.1. Run the installation script in a terminal

The installer can prompt for input and creates its own Python environment. Run it in a separate terminal so it does not inherit or modify the Jupyter environment.

In [ ]:
if DXCOM_PATH.is_file():
    print(f"DX-Compiler is already installed: {DXCOM_PATH}")
    print("Do not run the installer again.")
else:
    display(Markdown(
        "Run these commands in a separate terminal:\n\n"
        f"```bash\ncd {shlex.quote(str(DX_ALL_SUITE_DIR))}\n"
        "./dx-compiler/install.sh\n```"
    ))

After the DX-Compiler installation finishes successfully, a hint message is shown at the end.

The installer creates a dedicated Python virtual environment under `dx-compiler/venv-dx-compiler-local`. Keep it separate from the Jupyter virtual environment. The verification cells below call its executables with absolute paths, so they do not activate it inside the Notebook.

<pre style="color: green">
[HINT] ==================================================================== 
[HINT]   dx_com and dx_tron installation completed! 
[HINT]  
[HINT]   To use dx_com, activate the virtual environment first: 
[HINT]     $ source <path_to_dx_all_suite>/dx-compiler/venv-dx-compiler-local/bin/activate 
[HINT]  
[HINT]   Then you can run dxcom: 
[HINT]     $ dxcom -h 
[HINT]  
[HINT]   To run dxtron (no virtual environment required): 
[HINT]     $ dxtron 
[HINT]  
[HINT]   Or use the convenience script to start the web server: 
[HINT]     $ ./run_dxtron_web.sh --port=8080 
[HINT]  
[HINT] ==================================================================== 
</pre>

Next, verify the components installed by DX-Compiler:

In [ ]:
if DX_COMPILER_DIR.is_dir() and shutil.which("tree"):
    subprocess.run(["tree", "-L", "2", str(DX_COMPILER_DIR)], check=False)
else:
    print(f"DX-Compiler directory not ready: {DX_COMPILER_DIR}")

Now, let’s explore the files and folders under dx_com:

In [ ]:
if DX_COM_DIR.is_dir() and shutil.which("tree"):
    subprocess.run(["tree", "-L", "1", str(DX_COM_DIR)], check=False)
    sample_models = DX_COM_DIR / "sample_models"
    if sample_models.is_dir():
        subprocess.run(["tree", "-L", "2", str(sample_models)], check=False)
else:
    print(f"DX-COM directory not ready: {DX_COM_DIR}")

### 2.2. Verify DX-Compiler

In [ ]:
if not DX_COM_DIR.is_dir():
    raise FileNotFoundError(
        f"DX-COM is not installed at {DX_COM_DIR}. Run the terminal installation first."
    )
if not DXCOM_PATH.is_file():
    raise FileNotFoundError(f"dxcom executable not found: {DXCOM_PATH}")

print(f"DX-COM workspace: {DX_COM_DIR}")
print(f"dxcom executable: {DXCOM_PATH}")

Let's see help option information of `dxcom`:

In [ ]:
subprocess.run([str(DXCOM_PATH), "-h"], cwd=DX_COM_DIR, check=True)

Compile `MobileNetV1-1.onnx` to generate `MobileNetV1-1.dxnn` file:

In [ ]:
compile_result = subprocess.run(
    [
        str(DXCOM_PATH),
        "-m", "sample_models/onnx/MobileNetV2-1.onnx",
        "-c", "sample_models/json/MobileNetV2-1.json",
        "-o", "output/MobileNetV1-1",
        "--gen_log",
    ],
    cwd=DX_COM_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    check=False,
)
print(f"dxcom exit code: {compile_result.returncode}")

Check if `MobileNetV1-1.dxnn` file is generated:

In [ ]:
for path in (DX_COM_DIR / "sample_models", DX_COM_DIR / "output"):
    if path.exists() and shutil.which("tree"):
        subprocess.run(["tree", "-h", str(path)], check=False)
    else:
        print(f"Not found: {path}")

The `--gen_log` option outputs compilation logs to a file named `compiler.log`.

Let's see the stored log message:

In [ ]:
compiler_log = DX_COM_DIR / "output" / "MobileNetV1-1" / "compiler.log"
if compiler_log.is_file():
    print(compiler_log.read_text(encoding="utf-8", errors="replace"))
else:
    print(f"Compiler log not found: {compiler_log}")

### 2.3. DX-Tron

**DX-TRON** is a graphical visualization tool for exploring `.dxnn` model files compiled with the DEEPX toolchain. 

It allows users to load and inspect model structures, view workload distribution between NPU and CPU through color-coded graphs. 

With DX-TRON, users can better understand model execution flow and improve overall performance.

<img src="assets/sc-dxtron.png" style="max-width: 600px;">

**Key Features:**
- **Support for .dxnn Files**: Load and visualize model files compiled with the DEEPX toolchain.
- **Visual Workload Representation**: Displays a color-coded breakdown of workload execution:
- Red: Operations executed on the NPU
- Blue: Operations executed on the CPU or host
- **Model Navigation Controls**: Use the backward arrow in the bottom-left corner to return to the model overview screen at any time.
- **Interactive Node Inspection**: Double-click any node within the graph to view detailed information about the associated operations.
- FYI: DX-TRON was developed based on [netron](https://netron.app/) to support DXNN.

Let's run dxtron:
> `Note`: You can stop the `dxtron` by clicking the stop button ('■') above!

In [ ]:
dxtron = shutil.which("dxtron")
compiled_model = DX_COM_DIR / "output" / "MobileNetV1-1" / "MobileNetV2-1.dxnn"
if dxtron is None:
    print("dxtron is not installed or is not in PATH.")
elif not compiled_model.is_file():
    print(f"Compiled model not found: {compiled_model}")
else:
    subprocess.run([dxtron, str(compiled_model)], check=False)

## 3. Install DX-Runtime
For more details, see the [DX-All Suite installation guide](https://github.com/DEEPX-AI/dx-all-suite/blob/main-v2.3.3/docs/source/installation.md).

In [ ]:
if not (DX_ALL_SUITE_DIR / ".git").is_dir():
    raise FileNotFoundError(
        f"DX-All Suite repository not found: {DX_ALL_SUITE_DIR}. Run the terminal commands first."
    )

os.chdir(DX_ALL_SUITE_DIR)
print(f"DX-All Suite workspace: {Path.cwd()}")

### (Optional) Prerequisites before installation (`Orangepi-5 plus case` only)
 - If you use the official image of Orange Pi, the kernel header may not be installed. Kernel header is required to install the NPU driver.
 - Refer to the documentation linked [here](https://github.com/DEEPX-AI/dx-tutorials/blob/main/docs/orangepi5p.md).

### (Optional) Prerequisites before installation (`Raspberrypi-5 case` only)
 - PCIe is configured as Gen2 by default. It can be set to Gen3 to increase bandwidth. 
 - Refer to the documentation linked [here](https://github.com/DEEPX-AI/dx-tutorials/blob/main/docs/raspberrypi5.md). 

The DX-Runtime environment includes source code for each module. The repositories are managed as Git submodules(dx_rt_npu_linux_driver, dx_rt, dx_app, and dx_stream) under ./dx-runtime.

Let's see all options of DX-Runtime installation script:


In [ ]:
runtime_installer = DX_RUNTIME_DIR / "install.sh"
if runtime_installer.is_file():
    subprocess.run([str(runtime_installer), "--help"], cwd=DX_ALL_SUITE_DIR, check=False)
else:
    print(f"DX-Runtime installer not found: {runtime_installer}")

### 3.1. Install DX-Runtime in a terminal

The later tutorials use DX-APP and DX-STREAM, so the generated command uses `--all`. This full installation can take significantly longer because DX-APP supports more than 270 Model Zoo models.

> ./dx-runtime/install.sh --all

If you only need the core runtime, install the required targets instead:

> ./dx-runtime/install.sh --target=dx_rt_npu_linux_driver && ./dx-runtime/install.sh --target=dx_fw && ./dx-runtime/install.sh --target=dx_rt

Run the selected installation command in a separate terminal. The following cell prints the full-suite command only when DX-Runtime is missing.

In [ ]:
runtime_cli = shutil.which("dxrt-cli")
if runtime_cli:
    print(f"DX-Runtime is already installed: {runtime_cli}")
    print("Do not run the installer again.")
else:
    display(Markdown(
        "Run these commands in a separate terminal:\n\n"
        f"```bash\ncd {shlex.quote(str(DX_ALL_SUITE_DIR))}\n"
        "./dx-runtime/install.sh --all\n```\n\n"
        "A system reboot is required after the NPU driver installation. "
        "Return to this notebook after installation and reboot, then run the verification cell."
    ))

### 3.2. Verify the installation and save its location

The first verification checks the repository, submodules, DX-Compiler environment, and DX-Runtime CLI. When all required software checks pass, the selected installation path is saved to `dx-tutorials/config.json`. Hardware checks are reported separately below.

In [ ]:
verified_repo_exists = (DX_ALL_SUITE_DIR / ".git").is_dir()
verified_remote_url = ""
verified_branch_matches = False
verified_submodules_ready = False
if verified_repo_exists:
    _, verified_remote_url, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "remote", "get-url", "origin"]
    )
    _, verified_head, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "rev-parse", "HEAD"]
    )
    _, verified_expected, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "rev-parse", f"{DX_ALL_SUITE_BRANCH}^{{commit}}"]
    )
    verified_branch_matches = bool(verified_head) and verified_head == verified_expected
    status, verified_submodule_output, _ = command_output(
        ["git", "-C", str(DX_ALL_SUITE_DIR), "submodule", "status", "--recursive"]
    )
    verified_submodules_ready = status == 0 and all(
        not line.startswith("-") for line in verified_submodule_output.splitlines()
    )

repo_ok = verified_repo_exists and "DEEPX-AI/dx-all-suite" in verified_remote_url
compiler_ok = DXCOM_PATH.is_file()
runtime_cli = shutil.which("dxrt-cli")
runtime_ok = runtime_cli is not None

checks = {
    "DX-All Suite repository": repo_ok,
    f"Git branch {DX_ALL_SUITE_BRANCH}": verified_branch_matches,
    "Git submodules": verified_submodules_ready,
    "DX-Compiler environment": compiler_ok,
    "DX-Runtime CLI": runtime_ok,
}
for name, ready in checks.items():
    print(f"[{'OK' if ready else 'MISSING':7}] {name}")

software_verified = all(checks.values())
if software_verified:
    try:
        relative_to_home = DX_ALL_SUITE_DIR.relative_to(Path.home())
        saved_path = "~/" + relative_to_home.as_posix()
    except ValueError:
        saved_path = str(DX_ALL_SUITE_DIR)

    saved_config = {
        "schema_version": 1,
        "dx_all_suite_dir": saved_path,
        "git_branch": DX_ALL_SUITE_BRANCH,
    }
    CONFIG_PATH.write_text(
        json.dumps(saved_config, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"\nSaved DX-All Suite location to {CONFIG_PATH}")
else:
    print("\nInstallation is incomplete. config.json was not changed.")
    print("Complete the missing terminal installation steps and run this cell again.")

device_nodes = sorted(Path("/dev").glob("dxrt*"))
print(f"DEEPX devices: {device_nodes if device_nodes else 'not detected'}")

In [ ]:
# Check the PCIe generation and the number of lane for your NPU
# DX-M1 delivers the best performance on PCIe Gen3 (8GT/s), x4 lane - depending on your HOST system
!sudo lspci -vvv | grep -iA 33 accelerators | grep -E "LnkCap|LnkSta"

In [ ]:
# Check NPU kernel driver status -> expected: dxrt_driver, dx_dma
!lsmod | grep dxrt

In [ ]:
# Check NPU device files -> expected: '/dev/dxrt0'
!ls /dev/dxrt*

In [ ]:
# Check NPU daemon status -> expected: dxrt.service - DX-RT Service is active (running) state
!systemctl status dxrt.service

### 3.3. Useful tools provided by DX-RT

Let’s explore the executable CLI tools under dx_rt/bin:

In [ ]:
!ls dx-runtime/dx_rt/bin

#### 3.3.1. **dxbenchmark** is a CLI tool that executes a compiled .dxnn model to test functionality and measure performance.

In [ ]:
# Check dxbenchmark tool
!dxbenchmark -h

Download *.dxnn models and video files for quick validation.
> **Note**: If you remove the comment below and run `!cd dx-runtime/dx_app && bash setup.sh <<< ""`, it will download about 280 precompiled models, requiring around 20GB of storage.
> Unless necessary, it is recommended to download only the models you need.

In [ ]:
# Download *.dxnn models and video files for quick validation

# Note: 
#!cd dx-runtime/dx_app && bash setup.sh <<< ""

#!tree workspace/res

#### 3.3.2. **dxtop** s an htop-like tool for monitoring real-time DEEPX NPU metrics, such as utilization, temperature and memory.

> The output format from dxtop is not displayed correctly in the Jupyter Notebook code cell. **Open a separate terminal** to run the dxtop command instead. </br>
> How to open a separate termianl? `File >> New >> Terminal`
> 
> ![](assets/open-terminal.png)

> <img src="assets/sc-dxtop.png" style="max-width: 600px;">

In [ ]:
# Check dxtop tool
!dxtop -h
#!dxtop

#### 3.3.3. **dxrt-cli** is a command-line tool to query and monitor DEEPX DX-RT devices and read/wirte NPU firmware.

In [ ]:
# Check dxrt-cli tool
!dxrt-cli -h
#!dxrt-cli -s

In [ ]:
# Check the NPU F/W version
!dxrt-cli -i | grep FW

- Flashing NPU firmware image to DEEPX NPU
> `Note:` before the FW image flash, recommend to disable dxrt.service daemon

In [ ]:
# Stop dxrt.service daemon before the F/W image write
!sudo systemctl stop dxrt.service

In [ ]:
# NPU F/W image upgrade to 2.5.6 (latest)
!dxrt-cli -u dx-runtime/dx_fw/m1/latest/mdot2/fw.bin
#!dxrt-cli -u dx-runtime/dx_fw/m1/latest/h1/fw.bin
!sleep 5 # Delay for FW initializtion

In [ ]:
# Start dxrt.service daemon after the F/W image write
!sudo systemctl start dxrt.service

In [ ]:
# Check the NPU F/W version
!dxrt-cli -i | grep FW

#### 3.3.4. **run_model** is a CLI tool that executes a compiled .dxnn model to test functionality and measure performance.

In [ ]:
# Check run_model tool
!run_model -h
#!cd dx-runtime/dx_app && bash setup.sh
#!run_model -m workspace/res/models/YoloV5S_PPU.dxnn
#!run_model -m workspace/res/models/YoloV5S.dxnn

In [ ]:
!# Run DXTron
!dxtron \
     workspace/res/models/YOLOV5S_4.dxnn

YOLOV5S_4.dxnn consists of two parts - (NPU & CPU)

 ![](assets/sc-yolov5s.png)

If an operator is not supported by the NPU, it is processed on the CPU using ONNX Runtime via CPU offloading. 

When using run_model, adding the `--use-ort` option executes both the NPU and CPU components, allowing you to measure the performance of the entire .dxnn AI pipeline.

If the `--use-ort` option is omitted, performance measurement is limited to the NPU portion only.

In [ ]:
!echo "============================= NO ORT ================================="
!run_model -m workspace/res/models/YOLOV5S_4.dxnn -l 200 | grep FPS
!echo "======================================================================"
!echo ""
!echo "============================= USE ORT ================================"
!run_model -m workspace/res/models/YOLOV5S_4.dxnn -l 200 --use-ort | grep FPS
!echo "======================================================================"

#### 3.3.5. **parse_model** is a command-line tool that reads a compiled .dxnn model and displays its structure, inputs/outputs, and metadata.

In [ ]:
!parse_model -h
#!parse_model -m workspace/res/models/YOLOV5S_4.dxnn -v

## 4. Recap

<table align="left" border="1" style="width: 550px;">
  <tr style="background-color: #D0D0D0;"><th>Item</th><th>DX-Compiler</th><th>DX-Runtime</th></tr>
  <tr><td>Role</td><td>Model Compilation</td><td>Model Inference Execution</td></tr>
  <tr><td>Input</td><td>ONNX</td><td>DXNN</td></tr>
  <tr><td>Output</td><td>.dxnn</td><td>Inference Results</td></tr>
  <tr><td>System</td><td>x86_64 Only</td><td>x86_64, aarch64</td></tr>
  <tr><td>Auth Required</td><td>DEEPX Portal Account</td><td>Not Required</td></tr>
</table>